## Import libraries

In [84]:
import pandas as pd
import numpy as np

## Load  Data

In [85]:
df = pd.read_csv("../data/processed/cleaned_complaints.csv")

df.head()

,complaint_id,citizen_id,category,sub_category,ward,reported_date,status,people_affected,vulnerable_population,essential_service,public_safety_risk,recurrence_indicator,estimated_effort_hours,severity
0,C000001,P0869,Drainage,Drain Blockage,Peenya,2025-10-01,Closed,230.0,39.0,No,Medium,No,4.0,High
1,C000002,P0993,Sewerage,Sewage Overflow,Hennur,2025-10-01,Resolved,70.0,14.0,Yes,Critical,No,8.0,Medium
2,C000003,P1709,Street Infrastructure,Fallen Tree Branch,Dasarahalli,2025-10-01,Closed,190.0,12.0,No,Medium,No,12.0,Medium
3,C000004,P1458,Revenue Department,Trade Licence Issue,Chickpet,2025-10-01,Resolved,7.0,2.0,No,Low,No,2.0,Low
4,C000005,P1565,Water Supply,No Water Supply,Vasanthnagar,2025-10-01,Closed,140.0,21.0,Yes,Low,No,12.0,High


## View Data

In [86]:
df.shape

(8953, 14)

In [87]:
df.columns

Index(['complaint_id', 'citizen_id', 'category', 'sub_category', 'ward',
       'reported_date', 'status', 'people_affected', 'vulnerable_population',
       'essential_service', 'public_safety_risk', 'recurrence_indicator',
       'estimated_effort_hours', 'severity'],
      dtype='str')

In [88]:
df.dtypes

complaint_id                  str
citizen_id                    str
category                      str
sub_category                  str
ward                          str
reported_date                 str
status                        str
people_affected           float64
vulnerable_population     float64
essential_service             str
public_safety_risk            str
recurrence_indicator          str
estimated_effort_hours    float64
severity                      str
dtype: object

In [89]:
df.isnull().sum()

complaint_id                0
citizen_id                  0
category                    0
sub_category                0
ward                        0
reported_date               0
status                      0
people_affected             0
vulnerable_population       0
essential_service         119
public_safety_risk          0
recurrence_indicator        0
estimated_effort_hours    216
severity                    0
dtype: int64

In [90]:
print("Severity distribution:")
df["severity"].value_counts()

Severity distribution:


severity
Medium      3530
Low         2710
High        2000
Critical     713
Name: count, dtype: int64

In [91]:
"\nSeverity distribution in percentage:"
(df["severity"].value_counts(normalize=True) * 100).round(2)

severity
Medium      39.43
Low         30.27
High        22.34
Critical     7.96
Name: proportion, dtype: float64

## Remove columns that are not required for our ML project

In [92]:
df = df.drop(columns="essential_service")
df = df.drop(columns="estimated_effort_hours")

In [93]:
df.shape

(8953, 12)

In [94]:
df.columns

Index(['complaint_id', 'citizen_id', 'category', 'sub_category', 'ward',
       'reported_date', 'status', 'people_affected', 'vulnerable_population',
       'public_safety_risk', 'recurrence_indicator', 'severity'],
      dtype='str')

In [95]:
df.isnull().sum()

complaint_id             0
citizen_id               0
category                 0
sub_category             0
ward                     0
reported_date            0
status                   0
people_affected          0
vulnerable_population    0
public_safety_risk       0
recurrence_indicator     0
severity                 0
dtype: int64

In [96]:
df.head()

,complaint_id,citizen_id,category,sub_category,ward,reported_date,status,people_affected,vulnerable_population,public_safety_risk,recurrence_indicator,severity
0,C000001,P0869,Drainage,Drain Blockage,Peenya,2025-10-01,Closed,230.0,39.0,Medium,No,High
1,C000002,P0993,Sewerage,Sewage Overflow,Hennur,2025-10-01,Resolved,70.0,14.0,Critical,No,Medium
2,C000003,P1709,Street Infrastructure,Fallen Tree Branch,Dasarahalli,2025-10-01,Closed,190.0,12.0,Medium,No,Medium
3,C000004,P1458,Revenue Department,Trade Licence Issue,Chickpet,2025-10-01,Resolved,7.0,2.0,Low,No,Low
4,C000005,P1565,Water Supply,No Water Supply,Vasanthnagar,2025-10-01,Closed,140.0,21.0,Low,No,High


## New Features

### Feature 1: Vulnerable Population Ratio

Proportion of affected people who are vulnerable

In [97]:
df["vulnerable_population_ratio"] = np.where(
    df["people_affected"] > 0,
    df["vulnerable_population"] / df["people_affected"],
    0
)


In [98]:
df[
    [
        "people_affected",
        "vulnerable_population",
        "vulnerable_population_ratio"
    ]
].head(10)

,people_affected,vulnerable_population,vulnerable_population_ratio
0,230.0,39.0,0.169565
1,70.0,14.0,0.200000
2,190.0,12.0,0.063158
3,7.0,2.0,0.285714
4,140.0,21.0,0.150000
5,150.0,35.0,0.233333
6,119.0,7.0,0.058824
7,210.0,13.0,0.061905
8,55.0,0.0,0.000000
9,25.0,2.0,0.080000


In [99]:
df["vulnerable_population_ratio"].describe()

count    8953.000000
mean        0.119995
std         0.118162
min         0.000000
25%         0.039394
50%         0.088889
75%         0.166667
max         1.000000
Name: vulnerable_population_ratio, dtype: float64

In [100]:
df.groupby("severity")[
    [
        "people_affected",
        "vulnerable_population",
        "vulnerable_population_ratio"
    ]
].mean().round(2)


,people_affected,vulnerable_population,vulnerable_population_ratio
severity,,,
Critical,703.64,73.73,0.15
High,280.92,34.64,0.15
Low,145.67,9.62,0.09
Medium,225.14,21.72,0.12


### Feature 2: Complaint age

In [101]:
active_statuses = [
    "Registered",
    "Verified",
    "Assigned",
    "In Progress"
]


In [102]:
# Calculate complaint age using today's date

df["reported_date"] = pd.to_datetime(
    df["reported_date"],
    errors="coerce"
)

reference_date = pd.Timestamp.today().normalize()

In [103]:
df["complaint_age_days"] = np.where(
    df["status"].isin(active_statuses),
    (reference_date - df["reported_date"].dt.normalize()).dt.days,
    0
)

In [104]:
df[
    ["complaint_id", "reported_date", "status", "complaint_age_days"]
].head(10)

,complaint_id,reported_date,status,complaint_age_days
0,C000001,2025-10-01,Closed,0
1,C000002,2025-10-01,Resolved,0
2,C000003,2025-10-01,Closed,0
3,C000004,2025-10-01,Resolved,0
4,C000005,2025-10-01,Closed,0
5,C000006,2025-10-01,Resolved,0
6,C000007,2025-10-01,Resolved,0
7,C000008,2025-10-01,Resolved,0
8,C000009,2025-10-01,Closed,0
9,C000010,2025-10-01,Resolved,0


### Feature 3: Ward complaint load

In [105]:
df["is_active_complaint"] = df["status"].isin(active_statuses)

In [106]:
df["ward_complaint_load"] = (
    df["is_active_complaint"]
    .groupby(df["ward"])
    .transform("sum")
    .astype(int)
)

In [107]:
ward_load_summary = (
    df[["ward", "ward_complaint_load"]]
    .drop_duplicates()
    .sort_values("ward_complaint_load", ascending=False)
)

In [108]:
ward_load_summary.head(10)

,ward,ward_complaint_load
23,Mathikere,137
11,Hosahalli,85
2,Dasarahalli,67
62,Kadugodi,62
82,Sadashivanagar,60
13,Hoodi,50
159,Anjanapura,44
80,Bellandur,42
44,Shantinagar,38
6,Chamrajpet,35


## Verify the feature-engineered dataset


In [109]:
df.head()

,complaint_id,citizen_id,category,sub_category,ward,reported_date,status,people_affected,vulnerable_population,public_safety_risk,recurrence_indicator,severity,vulnerable_population_ratio,complaint_age_days,is_active_complaint,ward_complaint_load
0,C000001,P0869,Drainage,Drain Blockage,Peenya,2025-10-01,Closed,230.0,39.0,Medium,No,High,0.169565,0,False,25
1,C000002,P0993,Sewerage,Sewage Overflow,Hennur,2025-10-01,Resolved,70.0,14.0,Critical,No,Medium,0.200000,0,False,8
2,C000003,P1709,Street Infrastructure,Fallen Tree Branch,Dasarahalli,2025-10-01,Closed,190.0,12.0,Medium,No,Medium,0.063158,0,False,67
3,C000004,P1458,Revenue Department,Trade Licence Issue,Chickpet,2025-10-01,Resolved,7.0,2.0,Low,No,Low,0.285714,0,False,10
4,C000005,P1565,Water Supply,No Water Supply,Vasanthnagar,2025-10-01,Closed,140.0,21.0,Low,No,High,0.150000,0,False,7


In [110]:
df.shape

(8953, 16)

In [111]:
df.columns

Index(['complaint_id', 'citizen_id', 'category', 'sub_category', 'ward',
       'reported_date', 'status', 'people_affected', 'vulnerable_population',
       'public_safety_risk', 'recurrence_indicator', 'severity',
       'vulnerable_population_ratio', 'complaint_age_days',
       'is_active_complaint', 'ward_complaint_load'],
      dtype='str')

In [112]:
df.isnull().sum()

complaint_id                   0
citizen_id                     0
category                       0
sub_category                   0
ward                           0
reported_date                  0
status                         0
people_affected                0
vulnerable_population          0
public_safety_risk             0
recurrence_indicator           0
severity                       0
vulnerable_population_ratio    0
complaint_age_days             0
is_active_complaint            0
ward_complaint_load            0
dtype: int64

## Save the feature-engineered dataset

In [113]:
df.to_csv(
    "../data/processed/feature_engineered_complaints.csv",
    index= False
)